# Task 2: API REST — RAWG Video Games Database
### Evelyn Valeria Sarmiento Vásquez


In [1]:
!pip install requests pandas


In [ ]:
# Importando las librerías necesarias
import requests
import pandas as pd
from getpass import getpass

# getpass te pide la key como si fuera una contraseña — no queda guardada en el notebook
API_KEY = getpass("Ingresa tu RAWG API Key: ")

# URL base de la API — todas las consultas empiezan con esto
BASE_URL = "https://api.rawg.io/api"

#Al correr esto, va a aparecer un espacio para imputar mi api key como contraseña, es decir **********

print("API Key ingresada correctamente :)")


In [3]:
# Creamos un cliente que maneja todas las llamadas a la API
# Su función principal es dos cosas:
#   1. Agregar automáticamente tu API key a cada consulta (sin que tengas que escribirla cada vez)
#   2. Contar cuántas llamadas haces en total (para responder la pregunta D1)

class RAWGClient:
    def __init__(self, api_key):
        self.api_key = api_key
        self.total_requests = 0  # Contador de llamadas

    def get(self, endpoint, params={}):
        # Agregamos la API key a los parámetros automáticamente
        params["key"] = self.api_key
        url = f"{BASE_URL}/{endpoint}"
        response = requests.get(url, params=params)
        self.total_requests += 1  # Sumamos 1 al contador
        return response.json()

    def resumen_requests(self):
        print(f"Total de llamadas a la API realizadas: {self.total_requests}")

# Creamos el cliente listo para usar
client = RAWGClient(API_KEY)
print("Cliente RAWG listo ✓")


Cliente RAWG listo ✓


## Parte A — Exploración General

En esta sección consultamos la API de RAWG para obtener información general
sobre su base de datos de videojuegos.


In [4]:
# A1: Total de juegos registrados en RAWG
# El endpoint /games devuelve en el campo "count" el total de juegos en la base de datos

data = client.get("games")
total_games = data["count"]

print(f"RAWG tiene registrados {total_games:,} juegos en total.")


RAWG tiene registrados 898,445 juegos en total.


## Parte B — Análisis por Categorías

En esta sección analizamos los juegos mejor valorados según Metacritic,
y los mejores juegos disponibles en Steam.


In [5]:
# B1: Top 5 juegos con mayor puntaje Metacritic de todos los tiempos
# Ordenamos por metacritic de mayor a menor con el parámetro ordering=-metacritic

data = client.get("games", params={
    "ordering": "-metacritic",  # - significa descendente (mayor a menor)
    "page_size": 5              # Solo queremos los 5 primeros
})

print("🏆 Top 5 juegos mejor valorados por Metacritic:\n")
for i, game in enumerate(data["results"], start=1):
    print(f"{i}. {game['name']}")
    print(f"   Rating: {game['rating']} | Metacritic: {game['metacritic']}")
    print()


🏆 Top 5 juegos mejor valorados por Metacritic:

1. The Legend of Zelda: Ocarina of Time
   Rating: 4.38 | Metacritic: 99

2. Soulcalibur (1998)
   Rating: 0.0 | Metacritic: 98

3. Soulcalibur
   Rating: 4.38 | Metacritic: 98

4. Baldur's Gate III
   Rating: 4.44 | Metacritic: 97

5. Metroid Prime
   Rating: 4.35 | Metacritic: 97



In [6]:
# B2: Top 10 mejores juegos disponibles en Steam (store_id=1)
# Filtramos por la tienda Steam y ordenamos por metacritic

data = client.get("games", params={
    "stores": 1,                # store_id=1 es Steam
    "ordering": "-metacritic",
    "page_size": 10
})

print("🎮 Top 10 mejores juegos en Steam:\n")
for i, game in enumerate(data["results"], start=1):
    print(f"{i}. {game['name']}")
    print(f"   Rating: {game['rating']} | Metacritic: {game['metacritic']}")
    print()


🎮 Top 10 mejores juegos en Steam:

1. Baldur's Gate III
   Rating: 4.44 | Metacritic: 97

2. Half-Life 2: Update
   Rating: 4.13 | Metacritic: 96

3. Half-Life
   Rating: 4.38 | Metacritic: 96

4. Red Dead Redemption 2
   Rating: 4.59 | Metacritic: 96

5. Half-Life 2
   Rating: 4.48 | Metacritic: 96

6. BioShock
   Rating: 4.36 | Metacritic: 96

7. Grand Theft Auto IV: Complete Edition
   Rating: 4.57 | Metacritic: 95

8. Divinity: Original Sin 2
   Rating: 4.38 | Metacritic: 95

9. Portal 2
   Rating: 4.58 | Metacritic: 95

10. Red Dead Redemption
   Rating: 4.42 | Metacritic: 95



## Parte C — Comparaciones

En esta sección comparamos juegos por plataforma, géneros y años de lanzamiento.
